# 01 — LightGlue CUDA Graph Benchmark

Compare LightGlue (torch eager) vs CUDA Graph across batch sizes and point budgets.

**Key question:** How much does CUDA Graph remove kernel-launch overhead on Windows WDDM?

**Expected:** B=1/M=512/N=512: eager ~16ms → graph ~1.75ms (9× speedup)

**Dependencies:**
- A GPU with CUDA support (CUDA 11+)
- `accelerated_features` (for `LighterGlue` and `XFeatModel`)
- `kornia` (for LightGlue integration)
- `helper.py` from this repo

In [1]:
import sys, os, json, time, math
import numpy as np
import torch
import torch.nn.functional as F

# Relative path to project root (notebooks/ -> ../)
_PROJECT_ROOT = os.path.dirname(os.path.dirname(os.path.abspath("__file__"))) if "__file__" in dir() else os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.insert(0, _PROJECT_ROOT)

from helper import (
    patch_kornia_capture_safe,
    CGLightGlue,
    capture_cg_lightglue,
    replay_cg_lightglue,
)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"device = {device}")
if device == "cuda":
    print(f"GPU    = {torch.cuda.get_device_name(0)}")
    print(f"SM     = {torch.cuda.get_device_capability(0)}")

device = cuda
GPU    = NVIDIA GeForce RTX 4090 Laptop GPU
SM     = (8, 9)


In [2]:
# ── Build CGLightGlue model (einsum path, SDPA disabled) ──
patch_kornia_capture_safe()

model = CGLightGlue().to(device).eval()
MIN_CONF = model.net.conf.filter_threshold
print(f"LightGlue model loaded, n_layers={model.net.conf.n_layers}, "
      f"filter_threshold={MIN_CONF}")

# Quick smoke test
k0 = torch.zeros(1, 64, 2, device=device)
d0 = torch.zeros(1, 64, 64, device=device)
k1 = torch.zeros(1, 64, 2, device=device)
d1 = torch.zeros(1, 64, 64, device=device)
s0 = torch.tensor([[600, 400]], device=device, dtype=torch.long)
s1 = torch.tensor([[600, 400]], device=device, dtype=torch.long)
m0 = torch.zeros(1, 64, 1, device=device, dtype=torch.bool)
m1 = torch.zeros(1, 64, 1, device=device, dtype=torch.bool)
with torch.inference_mode():
    out = model(k0, d0, s0, k1, d1, s1, m0, m1, MIN_CONF)
print(f"Smoke test OK — output shapes: {[o.shape for o in out]}")

Loaded LightGlue model
LightGlue model loaded, n_layers=6, filter_threshold=0.1
Smoke test OK — output shapes: [torch.Size([1, 64]), torch.Size([1, 64]), torch.Size([1, 64]), torch.Size([1, 64])]


In [3]:
# ── Sweep parameters ──
BATCH_SIZES = [1, 2, 4, 8, 16]
POINT_BUDGETS = [64, 128, 256, 512]
WARM, REP = 10, 50

print(f"Sweep: B={BATCH_SIZES}  M=N={POINT_BUDGETS}")
print(f"Total configs: {len(BATCH_SIZES) * len(POINT_BUDGETS)}")

Sweep: B=[1, 2, 4, 8, 16]  M=N=[64, 128, 256, 512]
Total configs: 20


In [4]:
def make_dict_inputs(B, M, N, seed=42):
    """Generate B pairs of synthetic data as dicts."""
    np.random.seed(seed)
    d0_list, d1_list = [], []
    for b in range(B):
        n0 = np.random.randint(M // 2 + 1, M + 1) if M > 1 else 1
        n1 = np.random.randint(N // 2 + 1, N + 1) if N > 1 else 1
        d0_list.append({
            'keypoints': np.random.rand(n0, 2).astype(np.float32) * [600, 400],
            'descriptors': np.random.rand(n0, 64).astype(np.float32),
            'scores': np.random.rand(n0).astype(np.float32),
            'image_size': (640, 480),
        })
        d1_list.append({
            'keypoints': np.random.rand(n1, 2).astype(np.float32) * [600, 400],
            'descriptors': np.random.rand(n1, 64).astype(np.float32),
            'scores': np.random.rand(n1).astype(np.float32),
            'image_size': (640, 480),
        })
    return d0_list, d1_list


def dicts_to_tensors(d0_list, d1_list, B, M, N):
    """Convert dict format to eager batch tensors, sorting by score."""
    k0 = torch.zeros(B, M, 2, device=device)
    d0 = torch.zeros(B, M, 64, device=device)
    m0 = torch.zeros(B, M, 1, device=device, dtype=torch.bool)
    s0 = torch.zeros(B, 2, device=device, dtype=torch.long)
    k1 = torch.zeros(B, N, 2, device=device)
    d1 = torch.zeros(B, N, 64, device=device)
    m1 = torch.zeros(B, N, 1, device=device, dtype=torch.bool)
    s1 = torch.zeros(B, 2, device=device, dtype=torch.long)
    for b in range(B):
        n0 = min(len(d0_list[b]['scores']), M)
        n1 = min(len(d1_list[b]['scores']), N)
        if n0 > 0:
            idx = (-d0_list[b]['scores']).argsort()[:n0]
            k0[b, :n0] = torch.as_tensor(d0_list[b]['keypoints'][idx], device=device)
            d0[b, :n0] = torch.as_tensor(d0_list[b]['descriptors'][idx], device=device)
            m0[b, :n0, 0] = True
        s0[b] = torch.tensor(d0_list[b]['image_size'], dtype=torch.long, device=device)
        if n1 > 0:
            idx = (-d1_list[b]['scores']).argsort()[:n1]
            k1[b, :n1] = torch.as_tensor(d1_list[b]['keypoints'][idx], device=device)
            d1[b, :n1] = torch.as_tensor(d1_list[b]['descriptors'][idx], device=device)
            m1[b, :n1, 0] = True
        s1[b] = torch.tensor(d1_list[b]['image_size'], dtype=torch.long, device=device)
    return k0, d0, s0, k1, d1, s1, m0, m1

In [5]:
# ── Main benchmark loop ──
# Compare batch CUDA Graph vs batch eager (M=N only)
results = []
total = len(BATCH_SIZES) * len(POINT_BUDGETS)
done = 0

for B in BATCH_SIZES:
    for M in POINT_BUDGETS:
        N = M
        done += 1
        seed = 42 + B * 10000 + M

        # Build model + capture batch CUDA Graph
        model = CGLightGlue().to(device).eval()
        cg = capture_cg_lightglue(model, B=B, M=M, N=N, device=device)

        # Generate data once per config
        d0_list, d1_list = make_dict_inputs(B, M, N, seed=seed)
        tensors = dicts_to_tensors(d0_list, d1_list, B, M, N)
        # Build device-tensor dicts for the CUDA-Graph replay path.
        # replay_cg_lightglue now takes torch.Tensor inputs (kpts/desc/scores/size).
        def _to_cg_inputs(d):
            return {
                'keypoints': torch.as_tensor(d['keypoints'], device=device),
                'descriptors': torch.as_tensor(d['descriptors'], device=device),
                'scores': torch.as_tensor(d['scores'], device=device),
                'image_size': torch.tensor(d['image_size'], dtype=torch.long, device=device),
            }
        d0_t_list = [_to_cg_inputs(d0_list[b]) for b in range(B)]
        d1_t_list = [_to_cg_inputs(d1_list[b]) for b in range(B)]


        # Eager batch timing
        with torch.inference_mode():
            for _ in range(WARM):
                model(*tensors, MIN_CONF)
            torch.cuda.synchronize()
            el = []
            for _ in range(REP):
                t0 = time.perf_counter()
                model(*tensors, MIN_CONF)
                torch.cuda.synchronize()
                el.append((time.perf_counter() - t0) * 1000.0)
        el = np.array(el)

        # Batch CUDA Graph timing
        with torch.inference_mode():
            for _ in range(WARM):
                replay_cg_lightglue(cg, d0_t_list, d1_t_list, 0.1)
            torch.cuda.synchronize()
            gl = []
            for _ in range(REP):
                t0 = time.perf_counter()
                replay_cg_lightglue(cg, d0_t_list, d1_t_list, 0.1)
                torch.cuda.synchronize()
                gl.append((time.perf_counter() - t0) * 1000.0)
        gl = np.array(gl)

        rec = {
            "B": B, "M": M, "N": N,
            "eager_ms_p50": float(np.percentile(el, 50)),
            "eager_ms_mean": float(el.mean()),
            "graph_ms_p50": float(np.percentile(gl, 50)),
            "graph_ms_mean": float(gl.mean()),
            "speedup_p50": float(np.percentile(el, 50) / np.percentile(gl, 50)),
        }
        results.append(rec)

        del cg, model
        torch.cuda.empty_cache()

        print(f"[{done}/{total}] B={B} M={M} N={N}:  "
              f"eager={rec['eager_ms_p50']:.2f}ms  "
              f"graph={rec['graph_ms_p50']:.2f}ms  "
              f"speedup={rec['speedup_p50']:.2f}x", flush=True)

Loaded LightGlue model
[1/20] B=1 M=64 N=64:  eager=24.03ms  graph=2.44ms  speedup=9.83x
Loaded LightGlue model
[2/20] B=1 M=128 N=128:  eager=23.15ms  graph=2.55ms  speedup=9.10x
Loaded LightGlue model
[3/20] B=1 M=256 N=256:  eager=21.82ms  graph=2.09ms  speedup=10.46x
Loaded LightGlue model
[4/20] B=1 M=512 N=512:  eager=22.24ms  graph=2.56ms  speedup=8.69x
Loaded LightGlue model
[5/20] B=2 M=64 N=64:  eager=22.33ms  graph=3.50ms  speedup=6.38x
Loaded LightGlue model
[6/20] B=2 M=128 N=128:  eager=18.67ms  graph=3.12ms  speedup=5.98x
Loaded LightGlue model
[7/20] B=2 M=256 N=256:  eager=20.44ms  graph=3.42ms  speedup=5.97x
Loaded LightGlue model
[8/20] B=2 M=512 N=512:  eager=20.27ms  graph=4.62ms  speedup=4.39x
Loaded LightGlue model
[9/20] B=4 M=64 N=64:  eager=20.24ms  graph=4.14ms  speedup=4.89x
Loaded LightGlue model
[10/20] B=4 M=128 N=128:  eager=20.07ms  graph=4.21ms  speedup=4.76x
Loaded LightGlue model
[11/20] B=4 M=256 N=256:  eager=20.29ms  graph=5.02ms  speedup=4.04x
Lo

---
## Results summary

In [6]:
# ── Results summary ──
print("\n" + "=" * 70)
print("SUMMARY: LightGlue CUDA Graph vs Eager")
print("=" * 70)
print(f"{'B':>3} {'M':>4} {'N':>4}  {'eager(ms)':>10} {'graph(ms)':>10} {'speedup':>8}")
print("-" * 45)
for r in results:
    print(f"{r['B']:>3} {r['M']:>4} {r['N']:>4}  "
          f"{r['eager_ms_p50']:>8.2f}  {r['graph_ms_p50']:>8.2f}  "
          f"{r['speedup_p50']:>6.2f}x")


SUMMARY: LightGlue CUDA Graph vs Eager
  B    M    N   eager(ms)  graph(ms)  speedup
---------------------------------------------
  1   64   64     24.03      2.44    9.83x
  1  128  128     23.15      2.55    9.10x
  1  256  256     21.82      2.09   10.46x
  1  512  512     22.24      2.56    8.69x
  2   64   64     22.33      3.50    6.38x
  2  128  128     18.67      3.12    5.98x
  2  256  256     20.44      3.42    5.97x
  2  512  512     20.27      4.62    4.39x
  4   64   64     20.24      4.14    4.89x
  4  128  128     20.07      4.21    4.76x
  4  256  256     20.29      5.02    4.04x
  4  512  512     18.52      6.53    2.83x
  8   64   64     19.95      6.44    3.10x
  8  128  128     21.60      6.76    3.19x
  8  256  256     19.17      7.31    2.62x
  8  512  512     18.85     10.85    1.74x
 16   64   64     18.95      9.83    1.93x
 16  128  128     18.46     11.88    1.55x
 16  256  256     17.79     12.77    1.39x
 16  512  512     15.50     18.85    0.82x


In [7]:
# ── Best / worst cases ──
by_spd = sorted(results, key=lambda r: r['speedup_p50'])
print("\nWorst 3 speedups:")
for r in by_spd[:3]:
    print(f"  B={r['B']} M={r['M']} N={r['N']}:  "
          f"{r['eager_ms_p50']:.2f}ms → {r['graph_ms_p50']:.2f}ms  ({r['speedup_p50']:.2f}x)")

print("\nBest 3 speedups:")
for r in reversed(by_spd[-3:]):
    print(f"  B={r['B']} M={r['M']} N={r['N']}:  "
          f"{r['eager_ms_p50']:.2f}ms → {r['graph_ms_p50']:.2f}ms  ({r['speedup_p50']:.2f}x)")

print(f"\nFixed M=N=512, vary B:")
print(f"{'B':>3}  {'eager(ms)':>8} {'graph(ms)':>8} {'speedup':>8}")
for r in results:
    if r['M'] == 512 and r['N'] == 512:
        print(f"{r['B']:>3}  {r['eager_ms_p50']:>8.2f}  {r['graph_ms_p50']:>8.2f}  {r['speedup_p50']:>6.2f}x")


Worst 3 speedups:
  B=16 M=512 N=512:  15.50ms → 18.85ms  (0.82x)
  B=16 M=256 N=256:  17.79ms → 12.77ms  (1.39x)
  B=16 M=128 N=128:  18.46ms → 11.88ms  (1.55x)

Best 3 speedups:
  B=1 M=256 N=256:  21.82ms → 2.09ms  (10.46x)
  B=1 M=64 N=64:  24.03ms → 2.44ms  (9.83x)
  B=1 M=128 N=128:  23.15ms → 2.55ms  (9.10x)

Fixed M=N=512, vary B:
  B  eager(ms) graph(ms)  speedup
  1     22.24      2.56    8.69x
  2     20.27      4.62    4.39x
  4     18.52      6.53    2.83x
  8     18.85     10.85    1.74x
 16     15.50     18.85    0.82x
